# Reusable Template — Informal Lending Market Model

Swap in your own numbers to analyze a **different** informal lending scenario
(different fees, terms, products, or default assumptions) with the same NumPy
techniques from Chapter 3.

**Purpose:** this template is for **analysis and financial-literacy /
research use** — quantifying the true cost of informal credit and the risk
carried by an informal loan portfolio. It is explicitly **not** a tool for
designing loan terms, collection tactics, or a lending operation aimed at
extracting maximum interest from borrowers. See Section 5 before reusing it.

**To reuse this notebook for a new scenario:**
1. Edit only the `# >>> EDIT HERE <<<` cell in each of Sections 1, 2, and 4 —
   everything else is generic and runs unchanged.
2. Keep `terms`/`rates` and the obligation vector `b` the same length (an
   *n*-product mix matches an *n*-month obligation schedule).
3. Re-run the whole notebook (Kernel → Restart & Run All) after editing.

Read **Section 5 (Limitations)** — including the legal and ethical notes —
before drawing any conclusion from this template's output.


In [ ]:
import numpy as np
from numpy.linalg import solve

np.set_printoptions(precision=2, suppress=True)


## 1. True annualized cost of a short-term loan — edit this cell for your scenario

In [ ]:
# >>> EDIT HERE <<<
fee_rate = 0.10                 # flat fee per loan cycle (e.g. 0.10 = 10%)
terms_weeks = np.arange(1, 9)   # loan cycle lengths to compare, in weeks
# >>> END EDIT <<<

cycles_per_year = 52 / terms_weeks
EAR = (1 + fee_rate) ** cycles_per_year - 1

print("Effective annual rate by loan term:")
for t, ear in zip(terms_weeks, EAR):
    print(f"  {t}-week loan, {fee_rate*100:.0f}% flat fee -> EAR = {ear*100:8.1f}%")


## 2. Loan product mix — edit this cell for your scenario

`terms[j]` and `rates[j]` describe loan product `j` (term in months, flat
monthly rate). `obligation[i]` is the amount that must be collected in month
`i+1` to meet the lender's own funding commitment. All arrays must have
matching length — an *n*-product mix funds exactly *n* months of obligations.

In [ ]:
# >>> EDIT HERE <<<
terms = np.array([1, 2, 3])            # loan term in months, per product
rates = np.array([0.10, 0.08, 0.06])   # flat monthly rate, per product
obligation = np.array([5000, 4000, 3000])  # required monthly collections
# >>> END EDIT <<<

n = len(terms)
assert len(rates) == n and len(obligation) == n, \
    "terms, rates, and obligation must all be the same length"

A = np.zeros((n, n))
for i in range(1, n + 1):
    for j in range(n):
        if i <= terms[j]:
            A[i - 1, j] = 1 / terms[j] + rates[j]

b = obligation


## 3. Validity checks

Catch the most common reasons a cash-flow-matching problem is not solvable
*at all* — these do not tell you whether the scenario is realistic (see
Section 5), only whether the math is solvable.

In [ ]:
from numpy.linalg import matrix_rank

rank = matrix_rank(A)
if rank < n:
    raise ValueError(
        "The repayment matrix A is singular — cannot solve. This usually "
        "means two products have the same term, or a term falls outside "
        "1..n."
    )

if np.any(obligation <= 0):
    print("WARNING: an obligation value is <= 0 — check the `obligation` array.")

print("Validity checks passed. Repayment matrix A:\n", A)


## 4. Solve the product mix, then run the default-risk Monte Carlo

In [ ]:
x = solve(A, b)
print("Amount to lend via each product:", np.round(x, 2))
print("Check A @ x == b:", np.round(A @ x, 2))
print("Total capital deployed: $%.2f" % np.sum(x))


In [ ]:
def simulate_defaults(n_loans, principal, flat_rate, default_prob,
                       recovery_rate, n_scenarios=2000, seed=None):
    """Simulate `n_scenarios` portfolios of `n_loans` independent loans.
    Each loan either performs (repays principal*(1+flat_rate)) or defaults
    (recovers principal*recovery_rate). Returns total_collections, an array
    of length n_scenarios.
    """
    if seed is not None:
        np.random.seed(seed)
    performing_payoff = principal * (1 + flat_rate)
    defaulted_payoff = principal * recovery_rate

    defaults = np.random.random((n_scenarios, n_loans)) < default_prob
    collections_per_loan = np.where(defaults, defaulted_payoff, performing_payoff)
    return np.sum(collections_per_loan, axis=1)


# >>> EDIT HERE <<<
n_loans = 200
principal = 100
flat_rate = 0.10
default_prob = 0.15
recovery_rate = 0.30
n_scenarios = 2000
random_seed = 1
backer_obligation = 21000  # what must be repaid to the lender's own capital source
# >>> END EDIT <<<

total_collections = simulate_defaults(
    n_loans, principal, flat_rate, default_prob, recovery_rate,
    n_scenarios, random_seed
)

total_principal_lent = n_loans * principal
prob_loss = np.mean(total_collections < total_principal_lent)
prob_cant_repay_backer = np.mean(total_collections < backer_obligation)

print(f"Mean collections....: ${np.mean(total_collections):,.2f}")
print(f"Std deviation........: ${np.std(total_collections):,.2f}")
print(f"10th percentile......: ${np.percentile(total_collections, 10):,.2f}")
print(f"90th percentile......: ${np.percentile(total_collections, 90):,.2f}")
print(f"P(portfolio loses money): {prob_loss*100:.1f}%")
print(f"P(can't repay backer's ${backer_obligation:,}): {prob_cant_repay_backer*100:.1f}%")


## 5. Limitations — read before reusing on a new scenario

### ✅ Reasonable to use when
- Academic, policy, or financial-literacy analysis of **why** informal
  short-term credit is priced the way it is, and what risk a lender or a
  regulator should expect it to carry
- Teaching cash-flow matching (Section 2) and Monte Carlo default modeling
  (Section 4) as NumPy techniques, using illustrative, clearly-labeled
  numbers
- Helping a borrower or researcher see the **annualized** cost of a
  short-term loan quoted only as a flat per-cycle fee (Section 1) — this is
  a standard consumer-protection calculation, used to inform, not exploit

### 🚫 Not recommended when — and where this template stops on purpose
- **Designing loan terms, pricing, or collection strategy meant to extract
  maximum interest from borrowers.** This template computes costs and risk;
  it does not, and should not be extended to, recommend rates or tactics
  aimed at borrowers' disadvantage.
- **As a substitute for local legal advice.** Usury limits, licensing
  requirements, and consumer-protection law vary by jurisdiction and are not
  modeled here. A rate this notebook computes as mathematically consistent
  may be illegal to actually charge where you are.
- **As justification for real lending activity that harms borrowers.**
  Informal credit markets are associated with real risks — coercive
  collection, debt bondage, and cycles of over-indebtedness are documented
  problems in under-regulated informal lending. A default/recovery
  simulation cannot capture or excuse those harms.
- **Treating `fee_rate`, `default_prob`, `recovery_rate`, or `obligation`
  as measured facts about a real market.** They are placeholder assumptions.
  Real analysis needs real data (e.g., from household surveys, microfinance
  institution records, or published research), not invented numbers.
- **Any real decision by a borrower or a lending program** without
  professional, local guidance. This template supports understanding and
  research, not action.
